# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https:/|/another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-35B-A3B
Updated
6 days ago
•
769k
•
927
Qwen/Qwen3.5-9B
Updated
3 days ago
•
172k
•
389
unsloth/Qwen3.5-35B-A3B-GGUF
Updated
about 13 hours ago
•
674k
•
495
Qwen/Qwen3.5-27B
Updated
8 days ago
•
407k
•
571
Qwen/Qwen3.5-0.8B
Updated
2 days ago
•
93.4k
•
232
Browse 2M+ models
Spaces
Running
on
Zero
Featured
330
Omni Video Factory
🏆
330
text to video, image to video, video extend
Running
on
Zero
Featured
1.81k
Qwen Image Multiple Angles 3D Camera
🎥
1.81k
Change the camera angle of a photo with AI
Running
on
Zero
MCP
1.07k
W

In [22]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. It serves as a central collaboration platform where the global machine learning community comes together to create, share, and innovate on models, datasets, and AI applications. With over 2 million models, 500k datasets, and 1 million applications available, Hugging Face is the home of open-source AI development and an incubator of next-generation machine learning solutions across modalities like text, image, video, audio, and even 3D.

---

## What We Offer

- **Model Hub:** Explore and collaborate on 2M+ machine learning models contributed by the community.
- **Datasets:** Access and share over 500k datasets supporting diverse AI tasks.
- **Spaces:** Deploy and interact with AI apps in an intuitive online environment, fostering easy experimentation.
- **Open Source Stack:** Accelerate machine learning projects by leveraging our highly scalable open-source tools and libraries.
- **Enterprise Solutions:** Scalable, secure AI platform tailored for organizations with advanced security, analytics, and compute options.
- **Compute Resources:** Paid compute and enterprise-grade options like ZeroGPU to boost development speed and scalability.

---

## Enterprise & Team Solutions

Hugging Face provides tailored plans designed to support organizational needs, including:

- **Team Plans:** Starting at $20/user/month for small teams to collaborate securely and efficiently.
- **Enterprise Plans:** Customized contracts offering enhanced security with Single Sign-On (SSO), private storage, granular access controls, comprehensive audit logs, resource management, and priority customer support.
- **Advanced Compute Options:** Enhanced scalability with additional compute quotas and performance boosts.
- **Security Controls:** Organization-wide policies, billing management, and private data access for maximum control and compliance.

---

## Company Culture

Hugging Face champions an **open and ethical AI future**, fostering a vibrant and fast-growing community of machine learning engineers, scientists, and end users. The culture is deeply rooted in collaboration, transparency, and innovation, enabling creators worldwide to build their AI portfolios, share their work openly, and learn collectively. The company empowers diverse contributors by offering easy-to-use tools, pushing cutting-edge research and practical applications for real-world impact.

---

## Our Customers & Community

Hugging Face serves a wide spectrum of users — from independent researchers and developers to leading AI organizations. The platform is trusted by forward-thinking companies aiming to accelerate AI development securely and at scale. The open ecosystem supports AI research labs, educational institutions, startups, and enterprises, who use Hugging Face’s resources to build the next wave of intelligent applications.

---

## Careers & Opportunities

Hugging Face is continuously expanding and seeks passionate individuals eager to contribute to the AI revolution. Career opportunities focus on:

- Machine Learning Engineering
- AI Research
- Software Development
- Community Management
- Customer Success & Enterprise Sales

Working at Hugging Face means joining a mission-driven team dedicated to creating tools that empower the global AI community while adhering to principles of openness, inclusivity, and innovation.

---

## Join the Movement

Become a part of the AI community that is actively building the future of machine learning. Whether you are an individual creator, a startup, or a large enterprise, Hugging Face offers the tools, infrastructure, and collaborative environment to accelerate your AI journey.

**Explore, Share, Collaborate — Together.**

[Sign Up](https://huggingface.co/join) | [Learn More](https://huggingface.co/enterprise) | [Browse Models & Datasets](https://huggingface.co/models)

---

Hugging Face | The Home of Machine Learning  
Website: https://huggingface.co  
Contact: sales@huggingface.co

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [20]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face: The AI Community Building the Future

---

## About Us

Hugging Face is the premier collaboration platform for the global machine learning (ML) community. We serve as the central hub where ML engineers, data scientists, researchers, and AI enthusiasts come together to share, explore, discover, and experiment with open-source models, datasets, and applications. Our mission is to empower the next generation of machine learning professionals and end-users to build an open and ethical AI future collaboratively.

---

## Our Platform

- **Models:** Access and contribute to over 2 million state-of-the-art ML models spanning text, image, video, audio, and even 3D modalities.  
- **Datasets:** Explore and share more than 500,000 datasets curated from across the globe for training and benchmarking ML models.  
- **Spaces:** Deploy and interact with over 1 million applications, ranging from AI-powered video factories to text-to-speech generators and 3D camera angle changers.  
- **Community:** Join a fast-growing community passionate about collaboration, learning, and sharing. Build your professional portfolio by publishing your work for worldwide visibility.  
- **Enterprise & Compute Solutions:** Accelerate your ML workflows with scalable paid compute power and tailored enterprise solutions designed for teams and organizations.

---

## Why Choose Hugging Face?

- **Open Source Commitment:** Our platform is built on open-source technologies that enable faster experimentation and development cycles.  
- **Multi-Modality Support:** Work across modalities including text, image, video, audio, and 3D – all in one place.  
- **Collaborative Environment:** Host and collaborate on unlimited public projects to harness collective intelligence and innovation.  
- **Build Your Profile:** Showcase your machine learning expertise and projects, connecting you to a vibrant community and potential collaborators.  

---

## Company Culture

At Hugging Face, we believe in the power of community and transparency to drive AI innovation. We prioritize openness, collaboration, and ethical practices in AI development. Our culture welcomes diverse voices and encourages continuous learning and growth. We champion creating AI technologies that are accessible, trustworthy, and beneficial for all.

---

## Our Customers and Users

Our platform is used by:

- Independent ML researchers and enthusiasts  
- Data scientists and AI engineers in academia and industry  
- Startups and innovation labs pushing AI boundaries  
- Large enterprises deploying scalable AI solutions  
- Educational institutions integrating practical AI learning

They rely on Hugging Face for easy access to cutting-edge models and datasets, collaborative tools, and a community devoted to advancing AI together.

---

## Careers at Hugging Face

Join us if you are passionate about AI and eager to:

- Contribute to open-source machine learning projects impacting a global community  
- Work alongside leading experts in AI research and software development  
- Shape the future of responsible and ethical AI through innovation and collaboration  
- Thrive in a culture that values creativity, inclusion, and continuous improvement  

Explore opportunities across engineering, research, community management, and enterprise solutions. Become part of a team that's building the future of AI — together.

---

## Get Involved

- Browse models, datasets, and applications at [huggingface.co](https://huggingface.co)  
- Sign up to share your own ML projects and build your portfolio  
- Join community discussions and contribute to the open AI future  
- Leverage enterprise offerings for your organization’s AI needs  

---

**Hugging Face**  
*The Home of Machine Learning — Where Collaboration Powers Innovation*  
Visit us: [https://huggingface.co](https://huggingface.co)  

---

In [23]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 5 relevant links


# Hugging Face: Hugging the Future of AI, One Model at a Time

---

## Who Are We?  
Welcome to **Hugging Face**, the AI community where breakthrough machine learning isn’t just code — it’s a collaborative masterpiece! We’re the buzzing hub where ML engineers, data scientists, and AI enthusiasts mingle, share, and build the future together. From tiny text classifiers to massive 35-billion parameter models (yes, we’ve got those), you’ll find over **2 million models**, **500,000+ datasets**, and **1 million+ applications** ready to impress, transform, and innovate.

---

## What’s the Magic?  

- **Models, Models, Models:** Browse, test, and deploy everything from text-to-speech to 3D camera angle changers.  
- **Datasets Galore:** Raw materials for your AI recipes, updated constantly by a vibrant community.  
- **Spaces:** A magical place to showcase your AI demos and apps — basically your AI art gallery.  
- **Open Source for All:** We champion open, ethical AI — because making the future better is a team sport.  
- **Enterprise Ready:** Security, privacy, scalability — yep, the features big organizations need come standard.

---

## Why Hugging Face?  

- We’re *more* than a company, we're a thriving **community** — think of us as your AI family reunion that’s way less awkward and way more code.  
- Our open platform means your brilliant ideas aren’t stuck on your laptop — share, discover, collaborate, and maybe become an AI legend.  
- We believe *everyone* deserves a seat at the AI table, whether you’re a curious newbie or a wizard with 100+ models under your belt.  
- Ethical AI? Absolutely. We’re trailblazers dedicated to building tech that *helps* and inspires — not just disrupts.

---

## Culture: Where Algorithms Meet Awesomeness  

- People-powered innovation fueled by **collaboration**, **curiosity**, and an unapologetic love for machine learning.  
- Open arms for diverse talents, from coders to creative thinkers who dream in neural networks.  
- Work remotely or from wherever you feel inspired — freedom in how, when, and where you build.  
- A place where bugs get squashed, ideas get celebrated, and memes are part of the daily Scrum.  
- Competitive pay, growth opportunities, and the bragging rights to say you helped shape AI’s future.

---

## Want In? Careers at Hugging Face  

- Are you ready to join *the* AI revolution, possibly in roles like ML engineering, software development, research, or community engagement?  
- If your idea of fun includes deep dives into state-of-the-art AI models or building tools that *everyone* can use, you’ll feel right at home.  
- Bonus points if you enjoy a clever pun, appreciate well-placed emojis (🐍, 🤖, 💻, anybody?), or can teach a computer to laugh (we’re still working on that one).  

Check out our latest openings [here](#) and become part of the fun, smart, and sometimes quirky Hugging Face family.

---

## What Our Customers Say  

> *“Hugging Face didn’t just change how we build AI, it changed how we think about AI — from a solo feat to a community celebration.”* — Happy AI Engineer  

> *“Need open-source, ethical AI solutions with enterprise muscle? Hugging Face has you covered.”* — Large Enterprise CTO  

---

## Join Us — Let’s Build the Future, Together!  

Whether you want to crowdsource the next big model, explore cutting-edge datasets, or simply marvel at AI in action — Hugging Face welcomes dreamers, builders, and the curious. Because the future of AI is *hugely* collaborative, and it’s better with you here.

---

### Hugging Face - Where AI Gets a Warm Hug 🤗

**Sign up today and start hugging the future!**  
[Explore models](https://huggingface.co/models) | [Browse datasets](https://huggingface.co/datasets) | [Discover Spaces](https://huggingface.co/spaces) | [Join Careers](https://huggingface.co/careers)  

---

*P.S. Don't worry—no actual hugging required. AI hugs are way cooler.*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>